# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [5]:
#Сколько строк и столбцов? Какие типы у столбцов?
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 671.2 KB


In [9]:
# Какие типы у столбцов? Сколько пропусков в delivery_days?
print(df["delivery_days"].isna().sum())

160


In [14]:
# Сколько разных значений в `city`, `category`, `status`, `customer_id`?
len(df["city"].unique())

14

In [15]:
len(df["category"].unique())

5

In [16]:
len(df["status"].unique())

2

In [17]:
len(df["customer_id"].unique())

2003

#Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.
1. В городах есть повторения в названиях, но на разных языках, либо с пробелами или с заглавными буквами. Это вызовет проблема, потому что при подсчетах выручек или кол-ва будут некорректные числа, тем самым ошибочный анализ.
2. delivery_days есть пустые значения, которые также повлияют на конечный результат, к примеру если нужно вычислиить среднее значение дней доставок, то DataFrame просто первратить пустые дни None в 0 , но сам заказ войдет в общее кол-во и по итогу выйдет неточное среднее число.
3.  customer_id содержит тестовые данные, которые также повлияют на общий подсчет при анализе данных. Их надо исключить

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [29]:
paid = df["status"]=='оплачен'
paid

In [32]:
cust= df["customer_id"]!='TEST'
cust

0       True
1       True
2       True
3       True
4       True
        ... 
5458    True
5459    True
5460    True
5461    True
5462    True
Name: customer_id, Length: 5463, dtype: bool

In [34]:
paid_orders = df[paid&cust]
paid_orders

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
0,101674,2026-09-01 08:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен
1,101214,2026-09-01 08:00,C01210,Астана,Красота,3630.0,2.0,оплачен
2,102424,2026-09-01 08:00,C01408,Караганда,Дом,9120.0,3.0,оплачен
3,102333,2026-09-01 08:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен
4,102564,2026-09-01 09:00,C00657,Астана,Дом,11010.0,1.0,оплачен
...,...,...,...,...,...,...,...,...
5457,103557,2026-10-31 21:00,C01573,Алматы,Дом,9170.0,2.0,оплачен
5458,104168,2026-10-31 21:00,C01057,Алматы,Одежда,11430.0,2.0,оплачен
5459,104648,2026-10-31 21:00,C00629,Караганда,Книги,3640.0,2.0,оплачен
5460,104615,2026-10-31 22:00,C00557,Астана,Электроника,22150.0,NaN,оплачен


In [36]:
#среднее, медиана, 25-й и 75-й перцентили суммы.
paid_orders["amount"].mean()

np.float64(14672.391348704346)

In [38]:
paid_orders["amount"].median()

np.float64(10300.0)

In [39]:
np.percentile(paid_orders["amount"],25)

np.float64(5880.0)

In [40]:
np.percentile(paid_orders["amount"],75)

np.float64(18690.0)

In [42]:
#Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?
paid_orders["amount"].sum()

np.float64(71909390.0)

Типичный чек = это скорей всего средний чек заказов для понимания примерной выручки с одного заказа. Но правильные ориентироваться на percentile 50 либо mediana.
В отчете честнее показать медиану, потому что для вычисления медианы происходит сортировка и берется значение ровно посередине, что и будет верным показателем среднего чека, а если брать среднее значение, то оно может быть неточным в силу выбросов,  к примеру последние заказы могут быть с супер большими выручками, что в свою очередь очень сильно повлияет на средний чек.


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [81]:
cancelled_orders = df[df["status"]!='оплачен']
SH = ["Шымкент", "шымкент", "Shymkent"]
 
print(shym_orders[shym_orders["status"]!='оплачен']["amount"])
alm = df[
        (df["city"].str.lower().str.strip().isin(["almaty","алматы"]))
        & (df["status"]!='оплачен')
        ]
alm

5        3670.0
19      12540.0
62      21080.0
116     22450.0
174     13120.0
         ...   
5266     4220.0
5304    22610.0
5321    11420.0
5397    60910.0
5425     4890.0
Name: amount, Length: 164, dtype: float64


,order_id,order_date,customer_id,city,category,amount,delivery_days,status
10,100817,2026-09-01 09:00,C01934,Алматы,Электроника,24930.0,3.0,отменён
47,100347,2026-09-01 15:00,C01935,алматы,Одежда,12380.0,3.0,отменён
53,101136,2026-09-01 15:00,C01845,Алматы,Одежда,8450.0,2.0,отменён
87,100563,2026-09-01 21:00,C00572,Алматы,Одежда,10600.0,1.0,отменён
142,100570,2026-09-02 13:00,C01279,Алматы,Электроника,31220.0,1.0,отменён
...,...,...,...,...,...,...,...,...
5351,104485,2026-10-30 18:00,C00649,Алматы,Одежда,20590.0,3.0,отменён
5389,105118,2026-10-31 09:00,C01904,Алматы,Одежда,10100.0,3.0,отменён
5390,104512,2026-10-31 09:00,C00359,Алматы,Одежда,15890.0,3.0,отменён
5403,102931,2026-10-31 10:00,C01246,Алматы,Одежда,18830.0,3.0,отменён


In [82]:
print(f"Доля отменённых заказов: по всем заказам {round(cancelled_orders["amount"].sum()*100/df["amount"].sum())}%")
print(f"Доля отменённых заказов: в Шымкенте {round(shym_orders[shym_orders["status"]!='оплачен']["amount"].sum()*100/df["amount"].sum())}%")
print(f"Доля отменённых заказов: в Алматы {round(alm["amount"].sum()*100/df["amount"].sum())}%")

Доля отменённых заказов: по всем заказам 7%
Доля отменённых заказов: в Шымкенте 2%
Доля отменённых заказов: в Алматы 2%


и Шымкент и Алматы одинаковы в процентах отменненых заказов

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [108]:
import datetime as dt
shym_sep = df[(df["customer_id"] != "TEST")&(df["city"].isin(SH))&(df["status"]=="оплачен")&(pd.to_datetime(df["order_date"]).dt.month == 9)]["amount"].sum()
shym_oct = df[(df["customer_id"] != "TEST")&(df["city"].isin(SH))&(df["status"]=="оплачен")&(pd.to_datetime(df["order_date"]).dt.month == 10)]["amount"].sum()
alm_sep = df[(df["customer_id"] != "TEST")&(df["city"].str.strip().str.lower().isin(["almaty","алматы"]))&(df["status"]=="оплачен")&(pd.to_datetime(df["order_date"]).dt.month == 9)]["amount"].sum()
alm_oct = df[(df["customer_id"] != "TEST")&(df["city"].str.strip().str.lower().isin(["almaty","алматы"]))&(df["status"]=="оплачен")&(pd.to_datetime(df["order_date"]).dt.month == 10)]["amount"].sum()
print(f"Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте, изменение в {round(shym_oct/shym_sep*100 - 100)}%")
print(f"Выручка оплаченных заказов без TEST за сентябрь и октябрь в Алматы, изменение в {round(alm_oct/alm_sep*100 - 100)}%")

Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте, изменение в -18%
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Алматы, изменение в -6%


Падение сильнее в Шымкенте, в 3 раза сильнее чем в Алматы.
Менеджеру нужно сказать, что нужно обратить внимание на заказы в городе Шымкент, так как там падение почти 20 % за один месяц.

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [111]:
cancel_sh = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]
cancel_sh


,order_id,order_date,customer_id,city,category,amount,delivery_days,status
5,102231,2026-09-01 09:00,C00763,Шымкент,Книги,3670.0,1.0,отменён
19,100926,2026-09-01 11:00,C02131,Шымкент,Дом,12540.0,1.0,отменён
62,102059,2026-09-01 17:00,C00174,Шымкент,Дом,21080.0,3.0,отменён
116,100610,2026-09-02 10:00,C00101,Шымкент,Одежда,22450.0,2.0,отменён
174,100791,2026-09-02 17:00,C00188,Шымкент,Дом,13120.0,3.0,отменён
...,...,...,...,...,...,...,...,...
5266,103081,2026-10-29 16:00,C01415,Шымкент,Книги,4220.0,6.0,отменён
5304,104939,2026-10-30 09:00,C00147,Шымкент,Одежда,22610.0,7.0,отменён
5321,105133,2026-10-30 13:00,C00039,Шымкент,Одежда,11420.0,6.0,отменён
5397,104927,2026-10-31 10:00,C01361,Шымкент,Электроника,60910.0,7.0,отменён


1.Ошибка в том что условие внутри маски нужно передавать через скобки то есть [(condition) & (condition)]
2. То что в таблице город Шымкент сидит еще и как Shymkent и через маленькую букву шымкент. То есть cancel_sh не учитывает этот момент

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [126]:
top_customer = df["customer_id"].value_counts().idxmax()
orders_count = df["customer_id"].value_counts().max()

print(top_customer)
print(orders_count)

#df.loc[df["order_date","city","amount","status"]]
df.loc[
    df["customer_id"] ==top_customer, ["order_date","city","amount","status"]
]

C01818
11


,order_date,city,amount,status
57,2026-09-01 15:00,Шымкент,7350.0,оплачен
779,2026-09-09 18:00,Шымкент,45270.0,оплачен
1371,2026-09-15 21:00,Алматы,8890.0,оплачен
1375,2026-09-15 21:00,Алматы,8890.0,оплачен
2195,2026-09-24 18:00,Астана,1490.0,оплачен
2617,2026-09-29 09:00,Астана,22170.0,оплачен
2776,2026-09-30 21:00,Алматы,20180.0,оплачен
3417,2026-10-08 14:00,Караганда,4330.0,оплачен
4279,2026-10-18 08:00,Астана,13600.0,оплачен
4610,2026-10-21 19:00,Алматы,21990.0,оплачен


In [ ]:
#Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?
Да дубли имеются, выручка некорректная, нужно почистить дубли

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8